In [ ]:
# This subsection provides the necessary tool to produce uniform subsample given generated design and extracted features.

import numpy as np
from scipy.stats import norm
from sklearn.neighbors import NearestNeighbors

def design_transform(design):
    n, d = design.shape
    eta = np.random.choice(n, d)
    random_matrix = np.random.uniform(0, 1, (n, d))
    trans_design = design + (eta-0.5)/n
    RUD = trans_design % 1 + (random_matrix / n)
    RUD = RUD % 1
    return RUD

def split_indices(n,k):
    indices = np.random.permutation(n)
    size = int(n/k)
    split_i = []
    for i in range(k):
        if i < k-1:
            split_i.append(indices[i*size: (i+1)*size])
        else:
            split_i.append(indices[i*size: ])
    return(split_i)

#uniform selection for robust and uniform
def uniform_subsampling(data, design, k):
    # Data (N,D) is the extracted features for input images.
    # In our experiments, we use MoCo-v2 to extract features for images without labels.
    # The features correspond to the input before projection head.
    ## Design (n, D) is the uniform design generated using different methods. 
    ## Data and design should all be converted to numpy array so that the uniform subsample can be found.
    ## The function returns the indices for selected subsample. These indices should be saved in proper files.
    n, d = data.shape
    ind = split_indices(n, k)
    for j in range(k):
        ud = design_transform(design)
        neighbor = NearestNeighbors(n_neighbors=1, algorithm='kd_tree').fit(data[ind[j]])
        _, indices = neighbor.kneighbors(ud)
        indices_1d = np.squeeze(indices)
        pindices = ind[j][indices_1d.tolist()]
        if j == 0:
            total_indices = pindices
        else:
            total_indices = np.concatenate([total_indices, pindices])
    return total_indices

In [ ]:
# You could load a uniform design from given files
ud = np.loadtxt('./path/to/you/file', skiprows=1, delimiter=',')
ud = design_transform(ud)
# Or generate it using function provided below.
from UniformDesign import UniformDOE
ud=UniformDOE(1000,128,1000,100)
ud=(2*ud[1]-1)/2000

In [ ]:
# Once you obtain a uniform design, you should extract the features for candidate images.
# You should pass your hyperparameters for training MoCos on your own dataset. Below it is an example of our training setting. 
import model_cifar.resnet
import moco.builder
import moco.loader

model = moco.builder.MoCo("resnet50")
# We use the default hyperparameters setting for training MoCos on Camelyon17 dataset.
model.cuda()
# You should load your trained checkpoint which may be saved in some directory such as './checkpoint_best_camelyon17.pth.tar'
checkpoint = torch.load('./checkpoint_best_camelyon17.pth.tar', map_location = 'cuda:0')
model.load_state_dict(checkpoint["state_dict"])

# Extract features and convert them into numpy array for SimSRT selection.
features = torch.tensor([]).cuda()
for i, (image, _, _) in enumerate(train_loader):
    image = image.cuda()
    feat = model.encoder_q(image).data
    features = torch.cat((features,feat),dim=0)
M, _ = torch.max(features, 0)
m, _ = torch.min(features, 0)
features = (features - m )/(M-m)
features = features.cpu()
features = features.numpy()

In [ ]:
# Run uniform_subsampling to obtain the indices of robust subsample with size k*n, where n,d = ud.shape.
indices = uniform_subsampling(features, ud, k=5)
# Save the indices file for subsequent supervised training.
np.save("robust_indices.csv",indices,fmt="%d",delimiter=',',header='Indices')